In [1]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score


df = pd.read_csv("data/1/train.csv")

X = df.drop(columns=["Label"])
y = df["Label"]


pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("svm", SVC())
])

param_grid = {
    "svm__C": [0.1, 1, 10],
    "svm__kernel": ["linear", "rbf"],
    "svm__gamma": ["scale", 0.01, 0.001]
}


outer_cv = KFold(n_splits=10, shuffle=True, random_state=42)

results = []

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X), start=1):

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    inner_cv = KFold(n_splits=9, shuffle=True, random_state=42)

    grid = GridSearchCV(
        pipeline,
        param_grid,
        cv=inner_cv,
        scoring="f1_macro",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    preds = best_model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")

    results.append({
        "Fold": fold,
        "Accuracy": acc,
        "F1": f1,
        "Parameters": grid.best_params_
    })


results_df = pd.DataFrame(results)

print(results_df)

print("\nAccuracy mean ± std:",
      f"{results_df['Accuracy'].mean():.4f} ± {results_df['Accuracy'].std():.4f}")

print("F1 mean ± std:",
      f"{results_df['F1'].mean():.4f} ± {results_df['F1'].std():.4f}")


results_df.to_csv("svm_results.csv", index=False)


KeyboardInterrupt: 

In [11]:
import pandas as pd
df = pd.read_csv("data/9/train.csv")
print(df.shape)
print(df.dtypes.value_counts())
print(df.isnull().sum().sum(), "missing values")

(8330, 266)
float64    266
Name: count, dtype: int64
0 missing values


In [12]:
import os
print("CPU cores:", os.cpu_count())

import psutil
ram = psutil.virtual_memory()
print(f"RAM total: {ram.total / 1e9:.1f} GB")
print(f"RAM available: {ram.available / 1e9:.1f} GB")

CPU cores: 8
RAM total: 17.0 GB
RAM available: 5.6 GB


In [2]:
import pandas as pd, os, glob

files = glob.glob("results/phase1/svm/*_svm_summary.csv")
pd.concat([pd.read_csv(f) for f in files], ignore_index=True).to_csv("results/phase1/svm/all_datasets_svm_summary.csv", index=False)
print("Done")

Done


In [1]:
"""
Fills CS6735_PROJECT_-_PHASE_1.xlsx with Phase 1 results.
Reads from results/phase1/{clf}/dataset_{n}_{clf}_folds.csv for each classifier.
Outputs a filled xlsx to the same directory as this script.
"""

import os
import glob
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

# ── Config ─────────────────────────────────────────────────────────────────────

TEMPLATE   = "CS6735 PROJECT ORIGINAL.xlsx"
OUTPUT     = "CS6735_PROJECT_-_PHASE_1_FILLED.xlsx"
RESULTS    = "results/phase1"
N_DATASETS = 16
N_FOLDS    = 10

CLASSIFIERS = {
    "svm": {
        "acc_col": 2,   # B
        "f1_col":  3,   # C
        "par_col": 12,  # L
        "acc_key": "Accuracy",
        "f1_key":  "F1",
        "par_key": "Parameters",
    },
    "knn": {
        "acc_col": 4,
        "f1_col":  5,
        "par_col": 13,
        "acc_key": "Accuracy",
        "f1_key":  "F1",
        "par_key": "Parameters",
    },
    "dt": {
        "acc_col": 6,
        "f1_col":  7,
        "par_col": 14,
        "acc_key": "Accuracy",
        "f1_key":  "F1",
        "par_key": "Parameters",
    },
    "rf": {
        "acc_col": 8,
        "f1_col":  9,
        "par_col": 15,
        "acc_key": "Accuracy",
        "f1_key":  "F1",
        "par_key": "Parameters",
    },
    "mlp": {
        "acc_col": 10,
        "f1_col":  11,
        "par_col": 16,
        "acc_key": "Accuracy",
        "f1_key":  "F1",
        "par_key": "Parameters",
    },
}

# ── Load results ───────────────────────────────────────────────────────────────

def load_folds(clf: str, dataset_num: int) -> pd.DataFrame | None:
    path = os.path.join(RESULTS, clf, f"dataset_{dataset_num}_{clf}_folds.csv")
    if not os.path.exists(path):
        print(f"  [!] Missing: {path}")
        return None
    df = pd.read_csv(path)
    df = df.sort_values("Fold").reset_index(drop=True)
    return df


def load_summary(clf: str, dataset_num: int) -> pd.DataFrame | None:
    path = os.path.join(RESULTS, clf, f"dataset_{dataset_num}_{clf}_summary.csv")
    if not os.path.exists(path):
        return None
    return pd.read_csv(path)


# ── Find dataset block start rows ──────────────────────────────────────────────

def find_dataset_rows(ws) -> dict[int, int]:
    """
    Returns {dataset_number: first_fold_row} by scanning for 'Data N' headers.
    """
    dataset_rows = {}
    for row in ws.iter_rows():
        for cell in row:
            if cell.value and str(cell.value).strip().startswith("Data "):
                try:
                    num = int(str(cell.value).strip().replace("Data ", ""))
                    # First fold row is 2 rows below the Data N header
                    dataset_rows[num] = cell.row + 2
                except ValueError:
                    pass
    return dataset_rows


# ── Styles ─────────────────────────────────────────────────────────────────────

def mean_std_fill():
    return PatternFill("solid", start_color="D9EAD3")  # light green

def mean_std_font():
    return Font(bold=True)

thin = Side(style="thin")
def thin_border():
    return Border(left=thin, right=thin, top=thin, bottom=thin)


# ── Main ───────────────────────────────────────────────────────────────────────

def main():
    if not os.path.exists(TEMPLATE):
        raise FileNotFoundError(f"Template not found: {TEMPLATE}")

    wb = load_workbook(TEMPLATE)
    ws = wb.active

    dataset_rows = find_dataset_rows(ws)
    print(f"Found dataset blocks at rows: {dataset_rows}")

    for dataset_num in range(1, N_DATASETS + 1):
        if dataset_num not in dataset_rows:
            print(f"[!] Dataset {dataset_num} block not found in template, skipping")
            continue

        fold_start_row = dataset_rows[dataset_num]
        print(f"\nDataset {dataset_num} — fold rows {fold_start_row} to {fold_start_row + N_FOLDS - 1}")

        for clf, cfg in CLASSIFIERS.items():
            folds_df  = load_folds(clf, dataset_num)
            if folds_df is None:
                continue

            # Write fold values
            for fold_idx in range(N_FOLDS):
                row = fold_start_row + fold_idx
                fold_data = folds_df[folds_df["Fold"] == fold_idx + 1]
                if fold_data.empty:
                    print(f"  [!] {clf} dataset {dataset_num} fold {fold_idx+1} missing")
                    continue

                acc = fold_data[cfg["acc_key"]].values[0]
                f1  = fold_data[cfg["f1_key"]].values[0]
                par = fold_data[cfg["par_key"]].values[0]

                ws.cell(row=row, column=cfg["acc_col"]).value = round(float(acc), 4)
                ws.cell(row=row, column=cfg["f1_col"]).value  = round(float(f1),  4)
                ws.cell(row=row, column=cfg["par_col"]).value = str(par)

            # Write mean ± std rows (2 rows after last fold)
            summary_df = load_summary(clf, dataset_num)
            if summary_df is not None:
                mean_row = fold_start_row + N_FOLDS       # Mean row
                std_row  = fold_start_row + N_FOLDS + 1   # Std row

                acc_mean = summary_df["Accuracy Mean"].values[0]
                acc_std  = summary_df["Accuracy Std"].values[0]
                f1_mean  = summary_df["F1 Mean"].values[0]
                f1_std   = summary_df["F1 Std"].values[0]
                mean_std_acc = f"{acc_mean:.4f} ± {acc_std:.4f}"
                mean_std_f1  = f"{f1_mean:.4f} ± {f1_std:.4f}"

                for col in [cfg["acc_col"], cfg["f1_col"]]:
                    val = mean_std_acc if col == cfg["acc_col"] else mean_std_f1
                    cell = ws.cell(row=mean_row, column=col)
                    cell.value      = val
                    cell.fill       = mean_std_fill()
                    cell.font       = mean_std_font()
                    cell.alignment  = Alignment(horizontal="center")

            print(f"  ✓ {clf.upper()} written")

    wb.save(OUTPUT)
    print(f"\nSaved filled sheet → {OUTPUT}")

In [1]:
import pandas as pd

pairs = [(5,6), (7,8), (13,14), (15,16)]
for a, b in pairs:
    da = pd.read_csv(f"results/phase1/svm/dataset_{a}_svm_folds.csv")["Accuracy"].tolist()
    db = pd.read_csv(f"results/phase1/svm/dataset_{b}_svm_folds.csv")["Accuracy"].tolist()
    print(f"Data {a} vs {b}: identical={da==db}")

Data 5 vs 6: identical=True
Data 7 vs 8: identical=True
Data 13 vs 14: identical=True
Data 15 vs 16: identical=True


In [1]:
import pandas as pd

for i in range(9, 17):
    df = pd.read_csv(f"data/{i}/train.csv")
    vc = df["Label"].value_counts()
    print(f"\nDataset {i}: {df.shape}")
    print(vc)
    print(f"Dominant class %: {vc.iloc[0]/len(df)*100:.1f}%")


Dataset 9: (8330, 266)
Label
16.0    6764
3.0      430
6.0      233
5.0      195
7.0      156
4.0      127
1.0       85
2.0       81
11.0      46
8.0       44
9.0       38
15.0      34
10.0      27
14.0      27
13.0      26
12.0      17
Name: count, dtype: int64
Dominant class %: 81.2%

Dataset 10: (8330, 266)
Label
16.0    6770
3.0      402
6.0      252
5.0      182
7.0      174
4.0      131
1.0       95
8.0       55
10.0      46
15.0      44
2.0       38
9.0       35
13.0      29
12.0      28
14.0      25
11.0      24
Name: count, dtype: int64
Dominant class %: 81.3%

Dataset 11: (8330, 266)
Label
16.0    6832
6.0      297
3.0      165
1.0       95
5.0       92
11.0      88
4.0       87
14.0      80
8.0       79
9.0       78
12.0      75
13.0      75
2.0       73
10.0      73
15.0      71
7.0       70
Name: count, dtype: int64
Dominant class %: 82.0%

Dataset 12: (8330, 266)
Label
16.0    6763
12.0     125
1.0      121
10.0     119
9.0      114
8.0      112
13.0     105
15.0     104

In [2]:
pairs = [(5,6), (7,8), (13,14), (15,16)]
for a, b in pairs:
    da = pd.read_csv(f"data/{a}/train.csv")
    db = pd.read_csv(f"data/{b}/train.csv")
    print(f"Dataset {a} vs {b}: identical={da.equals(db)}")

Dataset 5 vs 6: identical=True
Dataset 7 vs 8: identical=True
Dataset 13 vs 14: identical=True
Dataset 15 vs 16: identical=True


In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import os

DATA_DIR = "data"
os.makedirs("report_figures", exist_ok=True)


def plot_group(datasets, title, output_path, color_map="Blues"):
    n    = len(datasets)
    cols = 4
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(18, 4 * rows))
    axes      = axes.flatten() if n > 1 else [axes]

    for i, ds in enumerate(datasets):
        path = os.path.join(DATA_DIR, str(ds), "train.csv")
        df   = pd.read_csv(path)
        vc   = df["Label"].value_counts().sort_index()

        n_classes = len(vc)
        colors    = plt.cm.get_cmap(color_map)(np.linspace(0.3, 0.85, n_classes))

        ax = axes[i]
        ax.bar(
            [str(int(c)) for c in vc.index],
            vc.values,
            color     = colors,
            edgecolor = "white",
            linewidth = 0.5,
        )

        ax.set_title(f"Dataset {ds}", fontsize=11, fontweight="bold", pad=6)
        ax.set_xlabel("Class", fontsize=9)
        ax.set_ylabel("Sample count", fontsize=9)
        ax.tick_params(axis="x", labelsize=8, rotation=30)
        ax.tick_params(axis="y", labelsize=8)
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.grid(axis="y", linestyle="--", alpha=0.4, linewidth=0.6)

        # Annotate dominant class
        dominant_idx = vc.values.argmax()
        ax.annotate(
            f"{vc.values[dominant_idx]:,}",
            xy         = (dominant_idx, vc.values[dominant_idx]),
            xytext     = (0, 4),
            textcoords = "offset points",
            ha         = "center",
            fontsize   = 7.5,
            fontweight = "bold",
            color      = "navy",
        )

    # Hide unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(title, fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: {output_path}")


# Group 1: Datasets 1-8 (4 classes, balanced)
plot_group(
    datasets    = range(1, 9),
    title = "Class Distribution - Datasets 1 to 8 (4 classes, moderate imbalance)",
    output_path = "report_figures/class_distribution_1_8.png",
    color_map   = "Blues",
)

# Group 2: Datasets 9-16 (16 classes, severe imbalance)
plot_group(
    datasets    = range(9, 17),
    title       = "Class Distribution - Datasets 9 to 16 (16 classes, severe imbalance)",
    output_path = "report_figures/class_distribution_9_16.png",
    color_map   = "RdYlGn",
)

C:\Users\akinb\AppData\Local\Temp\ipykernel_17832\1220834809.py:25: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors    = plt.cm.get_cmap(color_map)(np.linspace(0.3, 0.85, n_classes))


Saved: report_figures/class_distribution_1_8.png
Saved: report_figures/class_distribution_9_16.png


In [ ]:
import os
import pandas as pd

RESULTS_DIR = os.path.join("results", "final_test", "phase1")

files = [
    "all_phase1_svm_final_test_results.csv",
    "all_phase1_knn_final_test_results.csv",
    "all_phase1_dt_final_test_results.csv",
    "all_phase1_rf_final_test_results.csv",
    "all_phase1_mlp_final_test_results.csv",
]

frames = []
for file_name in files:
    path = os.path.join(RESULTS_DIR, file_name)
    if os.path.exists(path):
        frames.append(pd.read_csv(path))
    else:
        print(f"Missing: {path}")

combined = pd.concat(frames, ignore_index=True)
combined.to_csv(os.path.join(RESULTS_DIR, "all_phase1_final_test_results.csv"), index=False)

print("Rebuilt all_phase1_final_test_results.csv")